# EnvPool

> Steps several multi-agent envs together, as arrays of shape (num_envs, num_agents, ...).

In [ ]:
#| default_exp world.env_pool

In [ ]:
#| hide
from nbdev.showdoc import *

The multi-agent counterpart of stable-worldmodel's `EnvPool`. Envs keep their per-agent dict API;
the pool converts between the dicts and stacked arrays, so everything above it (`World`,
policies, datasets) only sees arrays with an `[env, agent, ...]` layout.

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Callable, Sequence

import numpy as np
from gymnasium import spaces

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.types import AgentID

In [ ]:
#| export
class EnvPool:
    """
    Runs ``num_envs`` multi-agent envs side by side, stacking their outputs as arrays of shape
    ``(num_envs, num_agents, ...)``.

    The envs are stepped one after the other in this process. :meth:`reset` and :meth:`step`
    take a `mask` to only reset / step some envs: the others keep their last observation and
    report zero reward and no termination.

    Array observations are written in place into buffers allocated on the first reset, so
    copy what you want to keep. Non-array observations (e.g. mission strings) are kept as
    nested lists ``[env][agent]``.
    """
    def __init__(self, env_fns: Sequence[Callable[[], MultiAgentEnv]]):
        self.envs: list[MultiAgentEnv] = [fn() for fn in env_fns]
        assert self.envs, "EnvPool needs at least one env"
        self.num_agents: int = self.envs[0].num_agents
        assert all(env.num_agents == self.num_agents for env in self.envs), \
            "All envs of a pool must have the same number of agents"
        self.seeds = np.full(self.num_envs, -1, dtype=np.int64)  # seed of each env's episode (-1: none)
        self._obs: dict[str, Any] | None = None

    @property
    def num_envs(self) -> int:
        return len(self.envs)

    @property
    def observation_space(self) -> spaces.Dict:
        "Observation space of one env (agent id -> agent observation space)."
        return self.envs[0].observation_space

    @property
    def action_space(self) -> spaces.Dict:
        "Action space of one env (agent id -> agent action space)."
        return self.envs[0].action_space

    @property
    def state_space(self) -> spaces.Space:
        "Global state space of one env."
        return self.envs[0].state_space

    def close(self):
        for env in self.envs:
            env.close()

In [ ]:
#| export
def _per_env(arg, num_envs: int, increment: bool = False) -> list:
    "One value per env: lists are used as is, an int seed becomes `seed + i`, anything else is shared."
    if isinstance(arg, (list, tuple, np.ndarray)):
        assert len(arg) == num_envs, f"Expected {num_envs} values, got {len(arg)}"
        return list(arg)
    if increment and isinstance(arg, (int, np.integer)):
        return [int(arg) + i for i in range(num_envs)]
    return [arg] * num_envs


def _is_array_like(value) -> bool:
    return isinstance(value, (np.ndarray, np.generic, bool, int, float)) and not isinstance(value, str)


def _allocate(obs: dict[AgentID, dict[str, Any]], num_envs: int) -> dict[str, Any]:
    "Buffers for the stacked observations, shaped like the per-agent observations of one env."
    num_agents, buffers = len(obs), {}
    for key, value in obs[0].items():
        if _is_array_like(value):
            value = np.asarray(value)
            buffers[key] = np.zeros((num_envs, num_agents, *value.shape), dtype=value.dtype)
        else:
            buffers[key] = [[None] * num_agents for _ in range(num_envs)]
    return buffers


def _to_action(action: np.ndarray):
    return action.item() if action.ndim == 0 else action

In [ ]:
#| export
from fastcore.utils import patch

@patch
def _indices(self: EnvPool, mask: np.ndarray | None) -> np.ndarray:
    return np.arange(self.num_envs) if mask is None else np.flatnonzero(mask)

@patch
def _write(self: EnvPool, i: int, obs: dict[AgentID, dict[str, Any]]):
    "Write the observations of env `i` into the stacked buffers."
    if self._obs is None:
        self._obs = _allocate(obs, self.num_envs)
    for a in range(self.num_agents):
        for key, value in obs[a].items():
            buffer = self._obs[key]
            if isinstance(buffer, np.ndarray):
                buffer[i, a] = value
            else:
                buffer[i][a] = value

@patch
def reset(
    self: EnvPool,
    seed: int | Sequence[int | None] | None = None,
    options: dict | Sequence[dict | None] | None = None,
    mask: np.ndarray | None = None) -> tuple[dict[str, Any], list[dict]]:
    """
    Reset the envs (only those where `mask` is True, if given).

    `seed` is an int (env `i` gets ``seed + i``), one seed per env, or None.
    `options` is one dict for every env, or one per env.
    Returns the stacked observations and the infos of each env.
    """
    if self._obs is None and mask is not None and not np.all(mask):
        raise RuntimeError("The first reset of an EnvPool must reset every env")
    seeds, options = _per_env(seed, self.num_envs, increment=True), _per_env(options, self.num_envs)
    infos = [{} for _ in range(self.num_envs)]
    for i in self._indices(mask):
        obs, infos[i] = self.envs[i].reset(seed=seeds[i], options=options[i])
        self.seeds[i] = -1 if seeds[i] is None else seeds[i]
        self._write(i, obs)
    return self._obs, infos

@patch
def step(
    self: EnvPool,
    actions: np.ndarray,
    mask: np.ndarray | None = None) -> tuple[dict[str, Any], np.ndarray, np.ndarray, np.ndarray, list[dict]]:
    """
    Step the envs (only those where `mask` is True, if given) with `actions` of shape
    ``(num_envs, num_agents, ...)``.

    Returns ``(observations, rewards, terminations, truncations, infos)``, where rewards,
    terminations and truncations have shape ``(num_envs, num_agents)``.
    """
    actions = np.asarray(actions)
    assert actions.shape[:2] == (self.num_envs, self.num_agents), \
        f"Expected actions of shape ({self.num_envs}, {self.num_agents}, ...), got {actions.shape}"
    shape = (self.num_envs, self.num_agents)
    rewards, terminations, truncations = np.zeros(shape, np.float32), np.zeros(shape, bool), np.zeros(shape, bool)
    infos = [{} for _ in range(self.num_envs)]
    for i in self._indices(mask):
        obs, rew, term, trunc, infos[i] = self.envs[i].step(
            {a: _to_action(actions[i, a]) for a in range(self.num_agents)})
        self._write(i, obs)
        for a in range(self.num_agents):
            rewards[i, a], terminations[i, a], truncations[i, a] = rew[a], term[a], trunc[a]
    return self._obs, rewards, terminations, truncations, infos

@patch
def state(self: EnvPool, mask: np.ndarray | None = None) -> np.ndarray:
    "Global states of the envs (only those where `mask` is True, if given), stacked on the first axis."
    return np.stack([self.envs[i].state() for i in self._indices(mask)])

def episode_done(terminations: np.ndarray, truncations: np.ndarray) -> np.ndarray:
    "Whether each env's episode is over: every agent is terminated or truncated. Shape (num_envs,)."
    return (terminations | truncations).all(axis=1)

In [ ]:
#| export
_all_ = ['episode_done']

### Tests

In [ ]:
from functools import partial
from stable_marl.envs import make

pool = EnvPool([partial(make, 'MultiGrid-Empty-6x6-v0', agents=2)] * 3)
obs, infos = pool.reset(seed=10)
assert pool.num_envs == 3 and pool.num_agents == 2 and list(pool.seeds) == [10, 11, 12]
assert obs['image'].shape == (3, 2, 7, 7, 3) and obs['direction'].shape == (3, 2)
assert obs['mission'].shape == (3, 2)   # missions are arrays (index in the MissionSpace)
assert pool.state().shape == (3, 6, 6, 3)

# the pool matches stepping the envs one by one
solo = [make('MultiGrid-Empty-6x6-v0', agents=2) for _ in range(3)]
solo_obs = [env.reset(seed=10 + i)[0] for i, env in enumerate(solo)]
rng = np.random.default_rng(0)
for _ in range(20):
    actions = rng.integers(3, size=(3, 2))
    obs, rew, term, trunc, _ = pool.step(actions)
    for i, env in enumerate(solo):
        o, r, te, tr, _ = env.step({a: int(actions[i, a]) for a in range(2)})
        for a in range(2):
            assert np.array_equal(obs['image'][i, a], o[a]['image']) and rew[i, a] == r[a] and term[i, a] == te[a]

# masked envs are not stepped
before = obs['image'].copy()
obs, rew, term, trunc, _ = pool.step(np.full((3, 2), 2), mask=np.array([True, False, False]))
assert np.array_equal(obs['image'][1:], before[1:]) and not rew[1:].any() and not term[1:].any()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()